In [3]:
import accelforge as af

# ── Configuration ──────────────────────────────────────────────────────────
CASES = {
    "Traditional (Dense, 128K)": {
        "arch": "archs/traditional.yaml",
        "mapping": "map_traditional_dense.yaml",
        "workload": "prob_dense.yaml",
    },
    "LongSight (Hybrid, 128K)": {
        "arch": "archs/longsight.yaml",
        "mapping": "map.yaml",
        "workload": "prob.yaml",
    },
}

# ── Run benchmarks ─────────────────────────────────────────────────────────
results = {}
for name, cfg in CASES.items():
    yaml_files = [cfg["arch"], cfg["mapping"], cfg["workload"]]
    spec = af.Spec.from_yaml(yaml_files)
    mappings = spec.map_workload_to_arch(
        print_progress=False,
        print_number_of_pmappings=False,
    )
    #mappings = mappings.prune_and_sort_by_energy()
    

    component_energy = mappings.energy(per_component=True)
    total_ops = mappings.actions(per_component=False)["compute"]
    latency = mappings.latency()
    print("total_ops:", total_ops)
    print("latency:", latency)
    total_energy_pj = sum(component_energy.values()) * 1e12
    energy_per_op = total_energy_pj / total_ops if total_ops else 0.0

    results[name] = {
        "latency": latency,
        "total_energy_pj": total_energy_pj,
        "total_ops": total_ops,
        "energy_per_op_pj": energy_per_op,
        "component_energy": {k: v * 1e12 for k, v in component_energy.items()},
    }

    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"{'=' * 60}")
    print(f"  Latency:          {latency:.6g}")
    print(f"  Total energy:     {total_energy_pj:.2f} pJ")
    print(f"  Compute ops:      {int(total_ops):,}")
    print(f"  Energy/op:        {energy_per_op:.4f} pJ/op")
    print(f"  Energy breakdown:")
    for comp, e_pj in sorted(component_energy.items(), key=lambda x: -x[1]):
        print(f"    {comp:30s} {e_pj * 1e12:12.2f} pJ")

# ── Summary comparison ─────────────────────────────────────────────────────
trad = results["Traditional (Dense, 128K)"]
ls = results["LongSight (Hybrid, 128K)"]

print(f"\n{'=' * 60}")
print(f"  Speedup & Savings")
print(f"{'=' * 60}")
if ls["latency"] > 0:
    print(f"  Latency ratio (Trad / LongSight):  {trad['latency'] / ls['latency']:.2f}x")
if ls["total_energy_pj"] > 0:
    print(f"  Energy  ratio (Trad / LongSight):  {trad['total_energy_pj'] / ls['total_energy_pj']:.2f}x")


total_ops: 268435456.0
latency: 0.19173961877822876

  Traditional (Dense, 128K)
  Latency:          0.19174
  Total energy:     120168372633.60 pJ
  Compute ops:      268,435,456
  Energy/op:        447.6621 pJ/op
  Energy breakdown:
    Trad_DRAM_Storage              120141529088.00 pJ
    Trad_TPU_Compute                26843545.60 pJ


ValidationError: 2 validation errors for Spec
arch.nodes.2
  Input tag 'Fanout' found using _get_tag() does not match any of the expected tags: 'Compute', 'Memory', 'Toll', 'Container', 'Network', 'Hierarchical', 'Array', 'Fork' [type=union_tag_invalid, input_value={'name': 'DReX_Package_Ar...may_reuse': 'Nothing'}]}, input_type=CommentedMap]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid
arch.nodes.4.Fork.nodes.0
  Input tag 'Fanout' found using _get_tag() does not match any of the expected tags: 'Compute', 'Memory', 'Toll', 'Container', 'Network', 'Hierarchical', 'Array', 'Fork' [type=union_tag_invalid, input_value={'name': 'DReX_ChannelBan...may_reuse': 'Nothing'}]}, input_type=CommentedMap]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid

In [3]:
import accelforge
print(accelforge.__version__)

AttributeError: module 'accelforge' has no attribute '__version__'

In [1]:
import matplotlib
print(f"Version: {matplotlib.__version__}")
print(f"Data Path: {matplotlib.get_data_path()}")

Version: 3.10.8
Data Path: /Users/arevalo/Documents/sp26/65930/project/.venv/lib/python3.13/site-packages/matplotlib/mpl-data


In [11]:
print(results)

{'Traditional (Dense, 128K)': {'latency': 0.191739611428571, 'total_energy_pj': 120141529088.000, 'total_ops': 268435456, 'energy_per_op_pj': 447.562072753906, 'component_energy': {'Trad_DRAM_Storage': 120141529088.000, 'Trad_TPU_Compute': 0}}, 'LongSight (Hybrid, 128K)': {'latency': 0.0153060282887289, 'total_energy_pj': 4998449920.00000, 'total_ops': 19575808, 'energy_per_op_pj': 255.338115290056, 'component_energy': {'HBM_Storage': 2896101376.00000, 'SM_Compute': 0, 'Bank_Storage': 2095688448.00000, 'PFU_Compute': 0, 'Query_SPM_Storage': 6660096.00000000, 'Address_SPM_Storage': 0, 'NMA_MAC_Compute': 0}}}


In [13]:
import matplotlib.pyplot as plt
import numpy as np

# Extraction
labels = list(results.keys())
short_labels = ['Traditional', 'LongSight'] # Shorter for X-axis clarity
energies = [results[l]["total_energy_pj"] for l in labels]
latencies = [results[l]["latency"] for l in labels]
epops = [results[l]["energy_per_op_pj"] for l in labels]

colors = ["#d62728", "#2ca02c"]
x = np.arange(len(labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Subplot 1: Total energy
axes[0].bar(x, energies, color=colors, width=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_labels, fontsize=10)
axes[0].set_ylabel("Energy (pJ)")
axes[0].set_title("Total Energy", fontweight='bold')
axes[0].ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

# Subplot 2: Latency
axes[1].bar(x, latencies, color=colors, width=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_labels, fontsize=10)
axes[1].set_ylabel("Latency (s)")
axes[1].set_title("Latency", fontweight='bold')
axes[1].ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

# Subplot 3: Energy per op
axes[2].bar(x, epops, color=colors, width=0.5)
axes[2].set_xticks(x)
axes[2].set_xticklabels(short_labels, fontsize=10)
axes[2].set_ylabel("pJ / op")
axes[2].set_title("Energy per Compute Op", fontweight='bold')

# Global Formatting
for ax in axes:
    ax.grid(axis="y", alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle("Traditional Dense vs. LongSight Hybrid — 128K Context Performance", 
             fontsize=15, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [ ]:
import matplotlib.pyplot as plt
import accelforge as af

TOTAL_KEYS = 130048
FILTERING_RATIOS = [5, 10, 20, 40]

spec = af.Spec.from_yaml(["archs/longsight.yaml", "map.yaml", "prob.yaml"])

sweep_energies = []
sweep_latencies = []
sweep_ops = []

print("Filtering Ratio Sweep (LongSight)\n")

for ratio in FILTERING_RATIOS:
    new_ks = int(TOTAL_KEYS / ratio)
    new_ks = max(8, (new_ks // 8) * 8)
    spec.workload.rank_sizes["K_s"] = new_ks

    print(f"  {ratio:>2d}x  (K_s={new_ks:>6d}) ... ", end="", flush=True)

    mappings = spec.map_workload_to_arch(
        print_progress=False,
        print_number_of_pmappings=False,
    )

    comp_energy = mappings.energy(per_component=True)
    energy_pj = sum(comp_energy.values()) * 1e12
    latency = mappings.latency()
    total_ops = mappings.num_computes()

    sweep_energies.append(energy_pj)
    sweep_latencies.append(latency)
    sweep_ops.append(total_ops)

    print(f"energy={energy_pj:.0f} pJ  latency={latency:.4g}  ops={int(total_ops):,}")
    for comp, e in sorted(comp_energy.items(), key=lambda x: -x[1]):
        print(f"        {comp:30s} {e * 1e12:12.2f} pJ")

# ── Plots ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(FILTERING_RATIOS, sweep_energies, "o-b")
ax1.set_xlabel("Filtering Ratio (x)")
ax1.set_ylabel("Total Energy (pJ)")
ax1.set_title("Energy vs. Filtering Ratio")
ax1.grid(True, alpha=0.3)

ax2.plot(FILTERING_RATIOS, sweep_latencies, "s-r")
ax2.set_xlabel("Filtering Ratio (x)")
ax2.set_ylabel("Latency")
ax2.set_title("Latency vs. Filtering Ratio")
ax2.grid(True, alpha=0.3)

plt.suptitle("LongSight: Effect of Filtering Ratio (128K Context)", y=1.02)
plt.tight_layout()
plt.show()
